In [1]:
RUN_MODE = "observed-dev"
CONTRACT_VERSION = "2.1.2"
CRAWL_RELEASE_ID = "CRAWL_20260806_03"
DATA_VERSION = "observed-dev-20260806.1"
AS_OF_DATE = "2026-08-06"
RANDOM_SEED = 42
DATA_PROVENANCE = "OBSERVED_DEVELOPMENT_ONLY"
EMPIRICAL_ANALYSIS_ALLOWED = False
PROMOTION_ALLOWED = False
DUTY_INPUT_PATH = ""
GOLD_INPUT_PATH = ""
CONTROL_SCHEMA_DIR = ""

# P4 Agent 4 · NCS Source Audit

**Stage:** `A4-00-NCS-SOURCE` · **Mode:** `observed-dev` · **Contract:** `2.1.2`

Audit 13,442 NCS units, checksum lineage, duplicates, levels, hierarchy codes, and documented name nulls.

> Development-only orchestration. Empirical analysis and production promotion are disabled.

In [2]:
from pathlib import Path
import os
import sys
import pandas as pd

NCS_ROOT = Path.cwd().resolve()
if NCS_ROOT.name != 'ncs_mapping':
    raise RuntimeError('run this notebook with cwd=ncs_mapping')
sys.path.insert(0, str(NCS_ROOT / 'src'))
assert RUN_MODE == 'observed-dev'
assert DATA_PROVENANCE == 'OBSERVED_DEVELOPMENT_ONLY'
assert EMPIRICAL_ANALYSIS_ALLOWED is False and PROMOTION_ALLOWED is False
resolved_duty_input = DUTY_INPUT_PATH or os.environ.get('P4_A2_DUTY_HANDOFF', '')
resolved_gold_input = GOLD_INPUT_PATH or os.environ.get('P4_NCS_GOLD_INPUT', '')
resolved_schema_dir = CONTROL_SCHEMA_DIR or os.environ.get('P4_CONTROL_SCHEMA_DIR', '')

In [3]:
from p4_ncs.quality.stage_artifacts import sha256_file

ncs_path = NCS_ROOT / 'data/processed/ncsUnit.parquet'
ncs_units = pd.read_parquet(ncs_path)
input_audit = {
    'rows': len(ncs_units),
    'parquetSha256': sha256_file(ncs_path),
    'duplicateNcsUnitCode': int(ncs_units['ncsUnitCode'].duplicated().sum()),
    'levels': sorted(ncs_units['ncsLevel'].dropna().astype(int).unique().tolist()),
    'hierarchyCodeNulls': int(ncs_units[['majorCode','middleCode','minorCode','subCode']].isna().sum().sum()),
    'hierarchyNameNulls': int(ncs_units[['majorName','middleName','minorName','subName']].isna().sum().sum()),
    'rawSha256Distinct': int(ncs_units['rawSha256'].nunique(dropna=True)),
}
assert input_audit['rows'] == 13_442
assert input_audit['duplicateNcsUnitCode'] == 0
assert input_audit['levels'] == list(range(1, 9))
input_audit

{'rows': 13442,
 'parquetSha256': 'b5b02ea232295931f245c69c881046a813f04cfbc517a13bc40d9e82994277c1',
 'duplicateNcsUnitCode': 0,
 'levels': [1, 2, 3, 4, 5, 6, 7, 8],
 'hierarchyCodeNulls': 0,
 'hierarchyNameNulls': 53768,
 'rawSha256Distinct': 1}

In [4]:
from p4_ncs.workflow.observed import run_stage

stage_manifest = run_stage('A4-00-NCS-SOURCE', root=NCS_ROOT, duty_input_path=resolved_duty_input or None, gold_input_path=resolved_gold_input or None, schema_dir=resolved_schema_dir or None)
stage_manifest

{'manifestVersion': 'stage-manifest-v1',
 'runId': 'NCS_MAPPING_OBSERVED_20260806_01',
 'runMode': 'observed-dev',
 'stageId': 'A4-00-NCS-SOURCE',
 'status': 'SUCCEEDED',
 'agentId': 'P4-A4-NCS',
 'branch': 'agent/p4-ncs-mapping-v2',
 'gitHead': 'cce6067e578cb8dc99aacaeb465439cb1ef0faa1',
 'contractVersion': '2.1.2',
 'schemaVersion': 'ncs-base-v1',
 'dataVersion': 'observed-dev-20260806.1',
 'crawlReleaseId': 'CRAWL_20260806_03',
 'dataProvenance': 'OBSERVED_DEVELOPMENT_ONLY',
 'startedAt': '2026-08-06T08:38:40.893217Z',
 'completedAt': '2026-08-06T08:38:40.894840Z',
 'empiricalAnalysisAllowed': False,
 'promotionAllowed': False,
 'inputManifestSha256': 'b5b02ea232295931f245c69c881046a813f04cfbc517a13bc40d9e82994277c1',
 'parameterSha256': 'ee2a95b4cf722fbc15def7f7c955eba2248f681ec2a54ffef33e1869c3836e36',
 'rowCounts': {'ncsUnit': 13442,
  'uniqueNcsUnitCode': 13442,
  'hierarchyCodeNulls': 0,
  'hierarchyNameNulls': 53768},
 'gateResults': [{'gateId': 'NCS_BASE_READY',
   'status': 

In [5]:
stage_root = NCS_ROOT / 'data/runs' / RUN_MODE / 'NCS_MAPPING_OBSERVED_20260806_01' / stage_manifest['stageId']
expected_artifacts = {'stage_manifest.json', 'stage_metrics.json', 'stage_quality.csv', 'CHECKSUMS.sha256'}
actual_artifacts = {path.name for path in stage_root.iterdir() if path.is_file()}
assert actual_artifacts == expected_artifacts
termination_summary = {'stageId': stage_manifest['stageId'], 'status': stage_manifest['status'], 'rowCounts': stage_manifest['rowCounts'], 'artifacts': sorted(actual_artifacts)}
termination_summary

{'stageId': 'A4-00-NCS-SOURCE',
 'status': 'SUCCEEDED',
 'rowCounts': {'ncsUnit': 13442,
  'uniqueNcsUnitCode': 13442,
  'hierarchyCodeNulls': 0,
  'hierarchyNameNulls': 53768},
 'artifacts': ['CHECKSUMS.sha256',
  'stage_manifest.json',
  'stage_metrics.json',
  'stage_quality.csv']}